# Continued Pretraining on Excel Spreadsheets

This notebook demonstrates how to use Training Hub's pretraining mode to do **continued pretraining (CPT)** on an instruct model using parsed Excel spreadsheet data.

**Pipeline:**
1. Download the [SpreadsheetBench](https://huggingface.co/datasets/KAKA22/SpreadsheetBench) dataset (real-world `.xlsx` files) from Hugging Face
2. Convert each spreadsheet to markdown text using Microsoft's [markitdown](https://github.com/microsoft/markitdown) library
3. Write a JSONL file with `{"document": "..."}` entries
4. Run `sft()` with `is_pretraining=True` for continued pretraining

**Model:** `ibm-granite/granite-3.3-8b-instruct`

**Why markdown tables?** Research shows that markdown table format significantly outperforms raw CSV for LLM comprehension of tabular data. The `markitdown` library naturally produces well-structured markdown tables with proper headers.

> See also: the script version at `examples/scripts/sft_cpt_spreadsheet_example.py`

## Setup and Imports

Install the required dependencies and import everything we need.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install training_hub 'markitdown[xlsx]' huggingface_hub

In [ ]:
import os
import sys
import json
import glob
import time
import tarfile
import logging
import warnings
import multiprocessing
from datetime import datetime

import torch
from huggingface_hub import hf_hub_download
from markitdown import MarkItDown

from training_hub import sft

# Required for CUDA multiprocessing in notebooks
multiprocessing.set_start_method("spawn", force=True)

# Suppress noisy openpyxl warnings about data validation extensions
warnings.filterwarnings("ignore", message="Data Validation extension")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPUs: {torch.cuda.device_count()}")
    print(f"GPU 0: {torch.cuda.get_device_name(0)}")

## Configuration

Set all paths and hyperparameters here. Adjust these to fit your hardware and experiment.

In [ ]:
# =============================================================================
# PATHS
# =============================================================================
model_path = "ibm-granite/granite-3.3-8b-instruct"  # HuggingFace model or local path
data_jsonl = "./spreadsheet_pretraining_data.jsonl"  # Output JSONL for training
cache_dir = "./spreadsheet_cache"                    # Cache for downloaded/extracted files
ckpt_output_dir = "./cpt_spreadsheet_checkpoints"    # Training checkpoint output

# =============================================================================
# DATASET
# =============================================================================
dataset_repo = "KAKA22/SpreadsheetBench"
dataset_file = "spreadsheetbench_verified_400.tar.gz"  # 395 verified real-world spreadsheets

# =============================================================================
# TRAINING HYPERPARAMETERS
# =============================================================================
num_epochs = 3
block_size = 2048           # Token block size for document packing
learning_rate = 2e-6        # Lower than SFT — we're injecting knowledge, not changing behavior
effective_batch_size = 64
max_tokens_per_gpu = 25000
max_seq_len = 4096
warmup_steps = 50

# =============================================================================
# HARDWARE
# =============================================================================
nproc_per_node = torch.cuda.device_count() if torch.cuda.is_available() else 1

print("Configuration:")
print(f"  Model:         {model_path}")
print(f"  Block size:    {block_size}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size:    {effective_batch_size}")
print(f"  Epochs:        {num_epochs}")
print(f"  GPUs:          {nproc_per_node}")

## Step 1: Download the SpreadsheetBench Dataset

The [SpreadsheetBench](https://huggingface.co/datasets/KAKA22/SpreadsheetBench) dataset contains real-world Excel spreadsheets collected from online Excel forums. We use the verified subset of ~400 spreadsheets.

The dataset contains pairs of files:
- `*_init.xlsx` — the original spreadsheet (input)
- `*_golden.xlsx` — the expected result after applying some operation

We only use the `_init.xlsx` files for pretraining.

In [ ]:
# Download the dataset from Hugging Face
tar_path = hf_hub_download(
    repo_id=dataset_repo,
    filename=dataset_file,
    repo_type="dataset",
    cache_dir=cache_dir,
)
print(f"Downloaded to: {tar_path}")

# Extract the archive
extract_dir = os.path.join(cache_dir, "extracted")
os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=extract_dir, filter="data")

# Find all _init.xlsx files (skip _golden.xlsx answer sheets)
all_xlsx = sorted(glob.glob(os.path.join(extract_dir, "**", "*.xlsx"), recursive=True))
xlsx_files = [f for f in all_xlsx if "_init.xlsx" in f] or all_xlsx

print(f"Extracted {len(all_xlsx)} total .xlsx files")
print(f"Using {len(xlsx_files)} input spreadsheets for training")

## Step 2: Convert Spreadsheets to Markdown Text

We use Microsoft's `markitdown` library to convert each `.xlsx` file into markdown text with proper table formatting. Each spreadsheet becomes one document in our training corpus.

Let's first look at what the conversion looks like for a single file:

In [ ]:
# Preview: convert one spreadsheet to see the output format
md = MarkItDown(enable_plugins=False)
sample_result = md.convert(xlsx_files[0])

print(f"File: {os.path.basename(xlsx_files[0])}")
print(f"Output length: {len(sample_result.text_content)} characters")
print()
print("--- Preview (first 800 chars) ---")
print(sample_result.text_content[:800])

In [ ]:
# Convert all spreadsheets and write to JSONL
md = MarkItDown(enable_plugins=False)
doc_count = 0
skip_count = 0

with open(data_jsonl, "w") as out:
    for i, xlsx_path in enumerate(xlsx_files, 1):
        try:
            result = md.convert(xlsx_path)
            text = result.text_content.strip()
            if text:
                out.write(json.dumps({"document": text}) + "\n")
                doc_count += 1
            else:
                skip_count += 1
        except Exception as e:
            print(f"  Skipping {os.path.basename(xlsx_path)}: {e}")
            skip_count += 1

        if i % 100 == 0:
            print(f"  Processed {i}/{len(xlsx_files)} files...")

print(f"\nConversion complete:")
print(f"  Documents written: {doc_count}")
print(f"  Files skipped:     {skip_count}")
print(f"  Output file:       {data_jsonl}")

### Inspect the Training Data

Let's verify the JSONL looks correct and get some basic statistics.

In [ ]:
# Load and inspect the generated JSONL
with open(data_jsonl) as f:
    docs = [json.loads(line) for line in f]

lengths = [len(d["document"]) for d in docs]

print(f"Total documents: {len(docs)}")
print(f"Character lengths:")
print(f"  Min:    {min(lengths):,}")
print(f"  Max:    {max(lengths):,}")
print(f"  Mean:   {sum(lengths)/len(lengths):,.0f}")
print(f"  Median: {sorted(lengths)[len(lengths)//2]:,}")
print(f"  Total:  {sum(lengths):,}")

# Show a snippet from the shortest and longest documents
shortest = min(docs, key=lambda d: len(d["document"]))
print(f"\n--- Shortest document ({len(shortest['document'])} chars) ---")
print(shortest["document"][:300])

## Step 3: Run Continued Pretraining

Now we run SFT in **pretraining mode** (`is_pretraining=True`). This tells Training Hub to:
- Treat each JSONL entry as a raw document (not chat messages)
- Pack documents into fixed-size blocks of `block_size` tokens
- Train with a causal language modeling objective

We use a conservative learning rate (`2e-6`) since we're doing continued pretraining on an already-trained instruct model — we want to inject spreadsheet knowledge without degrading existing capabilities.

In [ ]:
# Configure logging to show only essential information
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
)
# Reduce noise from libraries
for logger_name in ["torch", "transformers", "accelerate", "datasets"]:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
data_output_dir = f"data/cpt_spreadsheet_{timestamp}"

print("=" * 60)
print("Continued Pretraining: Excel Spreadsheets")
print("=" * 60)
print(f"  Model:              {model_path}")
print(f"  Data:               {data_jsonl}")
print(f"  Output:             {ckpt_output_dir}")
print(f"  GPUs:               {nproc_per_node}")
print(f"  Epochs:             {num_epochs}")
print(f"  Block size:         {block_size}")
print(f"  Batch size:         {effective_batch_size}")
print(f"  Learning rate:      {learning_rate}")
print(f"  Max seq len:        {max_seq_len:,}")
print(f"  Max tokens per GPU: {max_tokens_per_gpu:,}")
print()

start_time = time.time()

try:
    sft(
        # Model and data
        model_path=model_path,
        data_path=data_jsonl,
        ckpt_output_dir=ckpt_output_dir,

        # Pretraining mode
        is_pretraining=True,
        block_size=block_size,
        document_column_name="document",

        # Training parameters
        num_epochs=num_epochs,
        effective_batch_size=effective_batch_size,
        learning_rate=learning_rate,
        max_seq_len=max_seq_len,
        max_tokens_per_gpu=max_tokens_per_gpu,

        # Data processing
        data_output_dir=data_output_dir,
        warmup_steps=warmup_steps,
        save_samples=0,

        # Checkpointing
        checkpoint_at_epoch=True,
        accelerate_full_state_at_epoch=False,

        # Distributed setup
        nproc_per_node=nproc_per_node,
        nnodes=1,
    )

    duration = time.time() - start_time
    print("=" * 60)
    print(f"Training completed successfully!")
    print(f"  Duration: {duration/3600:.2f} hours")
    print(f"  Checkpoints: {ckpt_output_dir}/hf_format/")

except Exception as e:
    duration = time.time() - start_time
    print("=" * 60)
    print(f"Training failed after {duration/60:.1f} minutes")
    print(f"Error: {e}")
    print()
    print("Troubleshooting:")
    print("  - OOM? Try reducing max_tokens_per_gpu or block_size")
    print("  - Too few GPUs? Try increasing nproc_per_node")
    raise

## Step 4: Inspect the Checkpoint

Let's find the most recent checkpoint and verify it was saved correctly.

In [ ]:
# Find the most recent checkpoint
checkpoint_pattern = os.path.join(ckpt_output_dir, "hf_format", "samples_*")
checkpoint_dirs = sorted(glob.glob(checkpoint_pattern), key=os.path.getctime)

if checkpoint_dirs:
    latest_ckpt = checkpoint_dirs[-1]
    ckpt_files = os.listdir(latest_ckpt)
    print(f"Latest checkpoint: {latest_ckpt}")
    print(f"Files ({len(ckpt_files)}):")
    for f in sorted(ckpt_files):
        size_mb = os.path.getsize(os.path.join(latest_ckpt, f)) / (1024 * 1024)
        print(f"  {f:50s} {size_mb:8.1f} MB")
else:
    print("No checkpoints found.")

## Step 5: Visualize the Training Loss

Use Training Hub's built-in `plot_loss()` to visualize how the loss decreased during training.

In [ ]:
from training_hub import plot_loss

plot_loss(ckpt_output_dir, ema=True)

## Next Steps

After continued pretraining, you can:

1. **Evaluate** the model on spreadsheet-related tasks to measure knowledge acquisition
2. **Fine-tune** further with instruction data to create a spreadsheet-specialized assistant
3. **Merge** with the original model using linear interpolation to balance new knowledge with existing capabilities

The trained checkpoint is at the path printed above and can be loaded directly with `transformers`:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("./cpt_spreadsheet_checkpoints/hf_format/samples_XXXX")
tokenizer = AutoTokenizer.from_pretrained("./cpt_spreadsheet_checkpoints/hf_format/samples_XXXX")
```